# 03 — Semantic Basins on a Sentence Corpus
## BasinHop + ArrowSpace on sentence-level latent space

Notebook 01 established the synthetic baseline: ArrowSpace adds an orthogonal feature-manifold signal to vanilla minima search and improves local-minima quality on a controlled 3-cluster manifold.

Notebook 02 moved to a real embedding model and a real latent space: `all-MiniLM-L6-v2`, 384 dimensions, eight semantic fields, and a held-out set of polysemous probes. The main result was sharper and more semantically interpretable BasinHop minima under ArrowSpace augmentation.

This notebook is the next backlog step.

We move from isolated words to a **sentence corpus**. The goal is to test whether semantic basins remain meaningful when the unit of analysis is a short natural-language sentence rather than a single concept token.

Three hypotheses guide the notebook.

| ID | Hypothesis |
|---|---|
| H1 | Vanilla BasinHop on a sentence corpus still finds geometric minima, but basin membership becomes noisier because sentence-level variation adds syntactic and topical spread. |
| H2 | BasinHop + ArrowSpace recovers semantically purer sentence basins by preferring low-energy, feature-manifold-smooth regions in embedding space. |
| H3 | The blend coefficient `alpha` remains a useful control surface, and representative sentences extracted from the minima sets make the semantic effect directly inspectable. |

The augmented score is

`score_aug(x) = alpha * score_BH(x) + (1 - alpha) * lambda(x)`

where `score_BH` is the BasinHop / KDE-density score and `lambda` is the ArrowSpace feature-manifold energy map, both normalised to `[0, 1]`.

What this notebook adds beyond notebook 02:

- sentence-level corpus construction;
- optional real encoder embeddings or offline synthetic fallback;
- dual visualisation projections (PCA-2D and optional UMAP-2D);
- basin representative extraction;
- seed-stability analysis for minima overlap;
- richer artefacts saved to `output__03/`.


## 0. Setup

Dependencies: `arrowspace`, `sentence-transformers`, `scikit-learn`, `scipy`, `matplotlib`, and optionally `umap-learn`.

Like notebook 02, the notebook is CPU-friendly and includes an offline-safe fallback if model loading fails.


In [7]:
import os, warnings, json, math
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.stats import gaussian_kde

RNG = np.random.default_rng(42)
np.set_printoptions(precision=4, suppress=True)

os.makedirs("output__03", exist_ok=True)

# ── arrowspace (real library) or NumPy fallback ──────────────────────────────
USE_REAL_ARROWSPACE = True
GRAPH_PARAMS = None
try:
    from arrowspace import ArrowSpaceBuilder
    print('arrowspace available — using the real feature-space spectral engine.')
    GRAPH_PARAMS = {'eps': 2.0, 'k': 25, 'topk': 10, 'p': 2.0, 'sigma': 0.5}
except Exception as e:
    USE_REAL_ARROWSPACE = False
    print(f'arrowspace not importable ({e!r})\n-> using NumPy reference fallback.')
    GRAPH_PARAMS = {'eps': 0.5, 'k': 10, 'topk': 10, 'p': 2.0, 'sigma': 0.5}

try:
    import umap  # noqa: F401
    HAVE_UMAP = True
    print("umap-learn available.")
except Exception:
    HAVE_UMAP = False
    print("umap-learn not available; UMAP visualisation will be skipped.")

BASIN_PERCENTILE = 20
ALPHA_STEPS = np.round(np.arange(0.0, 1.05, 0.05), 2)
ALPHA_HIGHLIGHT = 0.35
KDE_BANDWIDTH = 0.30
N_STABILITY_SEEDS = 8
N_REPRESENTATIVES = 5


arrowspace available — using the real feature-space spectral engine.
umap-learn not available; UMAP visualisation will be skipped.


## 1. Build a labelled sentence corpus

We keep the eight semantic fields from notebook 02:

- astronomy
n- programming
- cooking
- finance
- emotions
- anatomy
- music
- geography

This time each field is represented by short declarative sentences. The corpus is deliberately small enough for CPU work, but rich enough to test whether sentence-level variation weakens or preserves basin structure.

A held-out boundary set of ambiguous or bridge-like sentences is kept for downstream probing, but not used to define fields.


In [8]:
FIELD_SENTENCES = {
    "astronomy": [
        "The telescope tracked a faint comet across the winter sky.",
        "Astronomers estimated the planet's orbit from repeated measurements.",
        "A nebula glowed behind the dense field of stars.",
        "The satellite adjusted its path after the eclipse window closed.",
        "Researchers measured parallax to estimate the star's distance.",
        "The observatory logged a burst of radiation from a pulsar.",
        "A meteor shower peaked just before dawn over the valley.",
        "The spacecraft photographed a cratered moon near the gas giant.",
        "Gravity bent the path of light around the massive object.",
        "The quasar appeared bright even at extreme distance.",
        "The constellation was visible above the horizon after sunset.",
        "The probe transmitted data from the edge of the magnetosphere."
    ],
    "programming": [
        "The compiler rejected the function because the types did not match.",
        "She refactored the module to reduce duplicated logic.",
        "A race condition appeared when two threads touched the same state.",
        "The debugger stopped at the line that mutated the array.",
        "The runtime cached the result to avoid repeated computation.",
        "A recursive function walked the tree until it found a leaf.",
        "The engineer pushed a patch after the failing test reproduced locally.",
        "The parser consumed tokens until it reached the closing brace.",
        "A pointer bug corrupted memory during the benchmark.",
        "The iterator yielded one record at a time from the stream.",
        "The service exposed a clean API for the client application.",
        "The deployment pipeline rolled back after the health check failed."
    ],
    "cooking": [
        "The chef simmered the broth until the flavours deepened.",
        "She whisked the sauce while the butter slowly melted.",
        "The dough rested before it went into the hot oven.",
        "A pinch of zest brightened the rich stew.",
        "They braised the vegetables until they turned soft and glossy.",
        "The pan was hot enough to sear the meat quickly.",
        "He julienned the carrots into thin even strips.",
        "The cook deglazed the skillet with white wine.",
        "Fresh herbs lifted the aroma of the roasted dish.",
        "The pastry needed another minute before the crust browned.",
        "She kneaded the dough until it felt smooth and elastic.",
        "The stockpot filled the kitchen with a savoury smell."
    ],
    "finance": [
        "The fund increased its hedge after market volatility returned.",
        "Rising inflation reduced the real yield on the bond.",
        "The analyst updated the valuation after the earnings call.",
        "The portfolio shifted toward higher-liquidity assets.",
        "A dividend increase signalled confidence from the board.",
        "The bank tightened collateral rules for new loans.",
        "Investors watched the maturity profile of the debt closely.",
        "The ledger showed a premium paid for the acquisition.",
        "The trader closed the arbitrage spread before the market moved.",
        "Cash flow improved after the company refinanced its liabilities.",
        "The repo market reflected short-term funding stress.",
        "A coupon payment arrived at the end of the quarter."
    ],
    "emotions": [
        "She felt a sudden wave of joy when the letter arrived.",
        "A quiet sense of dread settled over the empty room.",
        "His apology eased some of her lingering resentment.",
        "The reunion filled them with nostalgia and warmth.",
        "Anxiety made the wait feel much longer than it was.",
        "The painting inspired awe in almost every visitor.",
        "He spoke with affection even after the argument.",
        "Grief returned sharply at the sound of the old song.",
        "Their success brought relief more than excitement.",
        "Envy faded once she understood the work behind the result.",
        "The child looked at the stage with wonder.",
        "Contentment replaced the earlier tension by evening."
    ],
    "anatomy": [
        "The tendon connects muscle to bone at the joint.",
        "Signals travelled along the neuron toward the spinal cord.",
        "The surgeon examined the ventricle on the scan.",
        "Cartilage protected the knee from constant friction.",
        "The retina converts light into neural signals.",
        "Blood left the heart through the aorta.",
        "The cortex supports several higher cognitive functions.",
        "The ligament stabilised the ankle after the twist.",
        "The cornea refracts incoming light before it reaches the lens.",
        "Marrow inside the femur produces blood cells.",
        "The larynx controls airflow and helps generate speech.",
        "A synapse transmits information between neighbouring neurons."
    ],
    "music": [
        "The melody returned in a softer register near the end.",
        "A suspended chord delayed the harmonic resolution.",
        "The orchestra followed the conductor through the crescendo.",
        "Syncopation gave the rhythm a restless energy.",
        "The pianist shaped the phrase with delicate rubato.",
        "A low drone supported the vocal line underneath.",
        "The cadence landed cleanly in the final bar.",
        "Her vibrato widened during the sustained note.",
        "The fugue introduced each voice in careful sequence.",
        "Timbre mattered more than volume in the recording.",
        "The sonata opened with a tense repeated motif.",
        "A sudden diminuendo changed the emotional colour of the passage."
    ],
    "geography": [
        "The glacier carved a broad valley through the mountain range.",
        "Seasonal monsoon winds reshaped the coastal shoreline.",
        "The river widened into an estuary near the sea.",
        "A narrow isthmus joined the two larger landmasses.",
        "The plateau rose above the surrounding plain.",
        "Satellite maps traced the watershed across the region.",
        "The peninsula extended into the cold northern gulf.",
        "Heavy erosion deepened the canyon over time.",
        "The archipelago sits far beyond the main shipping route.",
        "Latitude strongly affects daylight in the winter months.",
        "The fjord cut inland between steep rocky slopes.",
        "Topography shaped settlement patterns across the basin."
    ]
}

BOUNDARY_SENTENCES = [
    "The bank of monitors showed a current of live market data.",
    "The bridge section modulated before the final chorus returned.",
    "A root process spawned another thread after the system reboot.",
    "The pitch of the proposal changed after the investor meeting.",
    "Mercury moved quickly across the morning sky above the harbour.",
    "The delta model shifted after new river measurements arrived.",
    "The cell line was stored beside the culture medium in the lab.",
    "Spring light changed the colour of the valley by noon.",
    "The key passage unlocked the argument in the final chapter.",
    "The scale of the map made the ridge look much smaller."
]

sentences, labels = [], []
for field, sents in FIELD_SENTENCES.items():
    for s in sents:
        sentences.append(s)
        labels.append(field)

LABELS = np.array(labels)
FIELD_NAMES = list(FIELD_SENTENCES.keys())
N_ITEMS = len(sentences)
print(f"Sentence corpus: {N_ITEMS} items across {len(FIELD_NAMES)} semantic fields.")
print("Items per field:", {k: len(v) for k, v in FIELD_SENTENCES.items()})
print(f"Held-out boundary sentences: {len(BOUNDARY_SENTENCES)}")


Sentence corpus: 96 items across 8 semantic fields.
Items per field: {'astronomy': 12, 'programming': 12, 'cooking': 12, 'finance': 12, 'emotions': 12, 'anatomy': 12, 'music': 12, 'geography': 12}
Held-out boundary sentences: 10


## 2. Encode the sentence corpus

We reuse the same encoder family as notebook 02 for continuity: `sentence-transformers/all-MiniLM-L6-v2`.

If model loading is unavailable, we synthesise a labelled sentence-level latent space with the same field structure plus small within-field variation.


In [9]:
def l2norm(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)


def load_latent_space(texts, extra_texts=None):
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
        X = model.encode(texts, normalize_embeddings=True, show_progress_bar=False).astype(np.float64)
        P = None
        if extra_texts:
            P = model.encode(extra_texts, normalize_embeddings=True, show_progress_bar=False).astype(np.float64)
        return X, P, "all-MiniLM-L6-v2 (sentence-transformers)", model
    except Exception as e:
        print(f"Real encoder unavailable ({e!r}) — synthesising sentence corpus latent space.")
        F = 384
        n_fields = len(FIELD_NAMES)
        centers = RNG.normal(size=(n_fields, F))
        X = []
        for lab in LABELS:
            c = centers[FIELD_NAMES.index(lab)]
            vec = c + 0.45 * RNG.normal(size=F)
            X.append(vec)
        X = l2norm(np.vstack(X).astype(np.float64))
        P = None
        if extra_texts:
            P = []
            for _ in extra_texts:
                i, j = RNG.integers(0, n_fields, size=2)
                vec = centers[i] + centers[j] + 0.6 * RNG.normal(size=F)
                P.append(vec)
            P = l2norm(np.vstack(P).astype(np.float64))
        return X, P, "synthetic sentence latent space", None


X, P_boundary, SOURCE, MODEL = load_latent_space(sentences, BOUNDARY_SENTENCES)
print(f"Latent space: {X.shape[0]} sentences, {X.shape[1]} dims, source = {SOURCE}")
if P_boundary is not None:
    print(f"Boundary set: {P_boundary.shape[0]} sentences, {P_boundary.shape[1]} dims")


Latent space: 96 sentences, 384 dims, source = all-MiniLM-L6-v2 (sentence-transformers)
Boundary set: 10 sentences, 384 dims


## 3. PCA-2D and optional UMAP-2D visual coordinates

As in notebook 02, PCA-2D is the primary deterministic visual coordinate system and the one used for the KDE density surface.

UMAP is added only as a secondary inspection view. It is useful for visual intuition, but all basin metrics still refer back to the original embedding space and the PCA/KDE basin score.


In [10]:
pca = PCA(n_components=2, random_state=42)
X2 = pca.fit_transform(X)
print(f"PCA explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%  "
      f"(PC1 {pca.explained_variance_ratio_[0]*100:.1f}%, PC2 {pca.explained_variance_ratio_[1]*100:.1f}%)")

if P_boundary is not None:
    X2_boundary = pca.transform(P_boundary)
else:
    X2_boundary = np.zeros((len(BOUNDARY_SENTENCES), 2))

if HAVE_UMAP:
    import umap
    um = umap.UMAP(n_neighbors=12, min_dist=0.15, metric="cosine", random_state=42)
    X_umap = um.fit_transform(X)
    X_umap_boundary = um.transform(P_boundary) if P_boundary is not None else np.zeros((len(BOUNDARY_SENTENCES), 2))
    print("UMAP projection ready.")
else:
    X_umap = None
    X_umap_boundary = None


PCA explained variance: 11.2%  (PC1 6.3%, PC2 4.9%)


## 4. Basin score in PCA-2D

We reuse the notebook 02 basin definition:

- fit a KDE over PCA-2D coordinates;
- define the vanilla BasinHop score as negative local density;
- lower score means denser and more central in the visualised latent surface.

This is still an approximation, but it gives a stable geometric baseline against which ArrowSpace can regularise the minima set.


In [11]:
kde = gaussian_kde(X2.T, bw_method=KDE_BANDWIDTH)

def density(pts):
    return kde(pts.T)

SBH = -density(X2)
SBH_NORM = (SBH - SBH.min()) / (SBH.max() - SBH.min() + 1e-12)
print(f"Basin score sBH: min={SBH.min():.4f}, max={SBH.max():.4f}")
print("Low score = high local density / geometric centrality in PCA-2D.")


Basin score sBH: min=-4.1628, max=-0.5762
Low score = high local density / geometric centrality in PCA-2D.


## 5. ArrowSpace feature-space Laplacian and energy map

We keep the same ArrowSpace construction as notebook 02:

- nodes are embedding dimensions, not items;
- the graph is built in feature space;
- each sentence receives a Rayleigh-style energy `lambda(x)`;
- low `lambda` means smoother, more typical, lower-curvature behaviour on the feature manifold.


In [12]:
def numpy_feature_laplacian(X, sigma=0.5, k=10):
    Fmat = X.T
    G = Fmat @ Fmat.T
    d2 = np.maximum(0.0, np.diag(G)[:, None] + np.diag(G)[None, :] - 2 * G)
    W = np.exp(-d2 / (2 * sigma ** 2))
    np.fill_diagonal(W, 0.0)
    nF = Fmat.shape[0]
    if k < nF - 1:
        drop = np.argsort(-W, axis=1)[:, k:]
        for i in range(nF):
            W[i, drop[i]] = 0.0
    W = np.maximum(W, W.T)
    deg = W.sum(1)
    Dinv = 1.0 / np.sqrt(deg + 1e-12)
    L = np.eye(nF) - Dinv[:, None] * W * Dinv[None, :]
    return L


def numpy_lambdas(X, sigma=0.5, k=10, eps=0.5):
    L = numpy_feature_laplacian(X, sigma=sigma, k=k)
    num = np.einsum("ij,jk,ik->i", X, L, X)
    den = np.einsum("ij,ij->i", X, X) + 1e-12
    R = np.abs(num / den)
    return R / (R.max() + eps)


def build_lambda_map(X):
    if USE_REAL_ARROWSPACE:
        builder = (ArrowSpaceBuilder()
            .with_seed(42)
            .with_dims_reduction(enabled=False, eps=None)
            .with_sampling("simple", 1.0))
        aspace, gl = builder.build(GRAPH_PARAMS, np.ascontiguousarray(X, dtype=np.float64))
        return np.asarray(aspace.lambdas(), dtype=float), aspace, gl
    return numpy_lambdas(X, sigma=GRAPH_PARAMS["sigma"], k=GRAPH_PARAMS["k"], eps=GRAPH_PARAMS["eps"]), None, None

LAM, HANDLE, GRAPH = build_lambda_map(X)
print(f"lambda-map: n={LAM.size}, min={LAM.min():.4f}, mean={LAM.mean():.4f}, max={LAM.max():.4f}")

if P_boundary is not None:
    X_all = np.vstack([X, P_boundary.astype(np.float64)])
    LAM_ALL, _, _ = build_lambda_map(X_all)
    LAM_BOUNDARY = LAM_ALL[N_ITEMS:]
    print(f"Boundary lambda mean={LAM_BOUNDARY.mean():.4f} vs clean corpus mean={LAM.mean():.4f}")
else:
    LAM_BOUNDARY = np.full(len(BOUNDARY_SENTENCES), LAM.mean())


lambda-map: n=96, min=0.0000, mean=0.1231, max=1.0000
Boundary lambda mean=0.1013 vs clean corpus mean=0.1231


## 6. Clustering and basin quality metrics in full embedding space

As in notebook 02, purity is measured in the original embedding space, not in PCA-2D.

We keep two simple quality metrics:

- **supervised purity**: plurality-label fraction inside a minima set;
- **semantic coherence**: mean pairwise cosine similarity inside the minima set.

We also extract the dominant label and representative sentences for each minima set.


In [13]:
K = len(FIELD_NAMES)
km = KMeans(n_clusters=K, random_state=42, n_init=20).fit(X)
KM_LABELS = km.labels_
print(f"k-means silhouette (k={K}): {silhouette_score(X, KM_LABELS):.4f}")
print(f"ground-truth silhouette: {silhouette_score(X, LABELS):.4f}")


def supervised_purity(idxs, label_arr=LABELS):
    if len(idxs) == 0:
        return 0.0
    vals, counts = np.unique(label_arr[idxs], return_counts=True)
    return counts.max() / len(idxs)


def dominant_label(idxs, label_arr=LABELS):
    if len(idxs) == 0:
        return None
    vals, counts = np.unique(label_arr[idxs], return_counts=True)
    return vals[np.argmax(counts)]


def mean_intra_cosine(idxs):
    if len(idxs) < 2:
        return 1.0
    V = X[idxs]
    sims = V @ V.T
    n = len(idxs)
    return (sims.sum() - n) / (n * (n - 1))


def field_composition(idxs):
    vals, counts = np.unique(LABELS[idxs], return_counts=True)
    order = np.argsort(-counts)
    return [(vals[i], int(counts[i])) for i in order]


k-means silhouette (k=8): 0.0728
ground-truth silhouette: 0.0716


## 7. Vanilla BasinHop minima and ArrowSpace-augmented minima

The minima set is the bottom `q = 20%` of the chosen score.

This notebook keeps the same interpretation:

- **vanilla** = geometry-first BasinHop/KDE surface;
- **augmented** = geometry regularised by feature-manifold smoothness.


In [14]:
def augmented_score(alpha, sbh_norm=SBH_NORM, lam=LAM):
    return alpha * sbh_norm + (1.0 - alpha) * lam


def minima_set(alpha, q=BASIN_PERCENTILE):
    s = augmented_score(alpha)
    thresh = np.percentile(s, q)
    return np.where(s <= thresh)[0], thresh


def eval_minima(idxs):
    return {
        "n": len(idxs),
        "purity": supervised_purity(idxs),
        "coherence": mean_intra_cosine(idxs),
        "meanlam": float(LAM[idxs].mean()) if len(idxs) else np.nan,
        "dominant": dominant_label(idxs),
    }

BH_MINIMA, BH_THRESH = minima_set(1.0)
AUG_MINIMA, AUG_THRESH = minima_set(ALPHA_HIGHLIGHT)

res_bh = eval_minima(BH_MINIMA)
res_aug = eval_minima(AUG_MINIMA)

print(f"Vanilla BasinHop minima (alpha=1.00, bottom {BASIN_PERCENTILE}%): {res_bh}")
print(f"ArrowSpace-augmented minima (alpha={ALPHA_HIGHLIGHT:.2f}, bottom {BASIN_PERCENTILE}%): {res_aug}")
print(f"Jaccard(BH, AUG) = {len(np.intersect1d(BH_MINIMA, AUG_MINIMA)) / len(np.union1d(BH_MINIMA, AUG_MINIMA)):.3f}")

print("Field composition — vanilla BH")
for name, count in field_composition(BH_MINIMA):
    print(f"  {name:12s} {count:2d} items ({100*count/len(BH_MINIMA):.0f}%)")

print("Field composition — augmented")
for name, count in field_composition(AUG_MINIMA):
    print(f"  {name:12s} {count:2d} items ({100*count/len(AUG_MINIMA):.0f}%)")


Vanilla BasinHop minima (alpha=1.00, bottom 20%): {'n': 20, 'purity': np.float64(0.4), 'coherence': np.float64(0.09638526330179481), 'meanlam': 0.059790383655046055, 'dominant': np.str_('anatomy')}
ArrowSpace-augmented minima (alpha=0.35, bottom 20%): {'n': 20, 'purity': np.float64(0.35), 'coherence': np.float64(0.09952915803430126), 'meanlam': 0.0297884549675959, 'dominant': np.str_('anatomy')}
Jaccard(BH, AUG) = 0.600
Field composition — vanilla BH
  anatomy       8 items (40%)
  emotions      4 items (20%)
  music         3 items (15%)
  programming   2 items (10%)
  astronomy     1 items (5%)
  cooking       1 items (5%)
  finance       1 items (5%)
Field composition — augmented
  anatomy       7 items (35%)
  music         4 items (20%)
  cooking       2 items (10%)
  emotions      2 items (10%)
  programming   2 items (10%)
  astronomy     1 items (5%)
  finance       1 items (5%)
  geography     1 items (5%)


## 8. Sweep `alpha` from 0 to 1

This is the same control surface used in notebooks 01 and 02.

For each `alpha`, we record:

- purity;
- coherence;
- mean `lambda`;
- dominant field;
- boundary-sentence membership count.


In [15]:
sbh_boundary_norm = (-density(X2_boundary) - SBH.min()) / (SBH.max() - SBH.min() + 1e-12) if P_boundary is not None else None

sweep_results = []
for alpha in ALPHA_STEPS:
    idxs, thresh = minima_set(alpha)
    r = eval_minima(idxs)
    r["alpha"] = float(alpha)
    r["threshold"] = float(thresh)
    if P_boundary is not None:
        s_aug_boundary = alpha * sbh_boundary_norm + (1.0 - alpha) * LAM_BOUNDARY
        r["n_boundary_in_minima"] = int((s_aug_boundary <= thresh).sum())
    else:
        r["n_boundary_in_minima"] = None
    sweep_results.append(r)

alphas = [r["alpha"] for r in sweep_results]
purities = [r["purity"] for r in sweep_results]
coherences = [r["coherence"] for r in sweep_results]
meanlams = [r["meanlam"] for r in sweep_results]
nboundary = [r["n_boundary_in_minima"] for r in sweep_results]
dom_fields = [r["dominant"] for r in sweep_results]

print(f"alpha sweep complete: {len(ALPHA_STEPS)} steps")
print(f"{'alpha':>6} {'purity':>7} {'coherence':>10} {'meanlam':>8} {'boundary':>9} {'dominant':>12}")
print("-" * 64)
for r in sweep_results:
    print(f"{r['alpha']:6.2f} {r['purity']:7.3f} {r['coherence']:10.4f} {r['meanlam']:8.4f} {str(r['n_boundary_in_minima']):>9} {str(r['dominant']):>12}")


alpha sweep complete: 21 steps
 alpha  purity  coherence  meanlam  boundary     dominant
----------------------------------------------------------------
  0.00   0.250     0.1050   0.0081         1      anatomy
  0.05   0.350     0.1013   0.0110         1      anatomy
  0.10   0.350     0.0921   0.0124         1      anatomy
  0.15   0.350     0.1017   0.0133         2      anatomy
  0.20   0.350     0.1017   0.0133         2      anatomy
  0.25   0.400     0.1021   0.0164         2      anatomy
  0.30   0.300     0.1035   0.0238         2      anatomy
  0.35   0.350     0.0995   0.0298         2      anatomy
  0.40   0.350     0.0995   0.0298         2      anatomy
  0.45   0.350     0.1060   0.0337         2      anatomy
  0.50   0.350     0.1060   0.0337         2      anatomy
  0.55   0.350     0.0991   0.0414         2      anatomy
  0.60   0.350     0.0991   0.0414         2      anatomy
  0.65   0.400     0.0988   0.0521         2      anatomy
  0.70   0.400     0.0988   0.0521

## 9. Representative sentences for each basin set

A sentence-level corpus lets us inspect minima directly.

For each minima set we extract the lowest-score representatives. This is the most concrete way to see whether augmentation produces cleaner semantic cores rather than merely shifting summary metrics.


In [16]:
def top_representatives(idxs, score, n=N_REPRESENTATIVES):
    order = idxs[np.argsort(score[idxs])[:n]]
    rows = []
    for i in order:
        rows.append({
            "idx": int(i),
            "label": LABELS[i],
            "sentence": sentences[i],
            "score": float(score[i]),
            "lambda": float(LAM[i]),
            "sbh": float(SBH_NORM[i]),
        })
    return pd.DataFrame(rows)

score_bh = augmented_score(1.0)
score_aug = augmented_score(ALPHA_HIGHLIGHT)

rep_bh = top_representatives(BH_MINIMA, score_bh)
rep_aug = top_representatives(AUG_MINIMA, score_aug)

print("Top representatives — vanilla BasinHop")
display(rep_bh)
print("Top representatives — ArrowSpace augmented")
display(rep_aug)

rep_bh.to_csv("output__03/representatives_bh.csv", index=False)
rep_aug.to_csv("output__03/representatives_aug.csv", index=False)


Top representatives — vanilla BasinHop


,idx,label,sentence,score,lambda,sbh
0,63,anatomy,Cartilage protected the knee from constant fri...,0.000000,0.010118,0.000000
1,71,anatomy,A synapse transmits information between neighb...,0.043521,0.083223,0.043521
2,60,anatomy,The tendon connects muscle to bone at the joint.,0.060641,0.113563,0.060641
3,35,cooking,The stockpot filled the kitchen with a savoury...,0.075118,0.092142,0.075118
4,49,emotions,A quiet sense of dread settled over the empty ...,0.081245,0.020659,0.081245


Top representatives — ArrowSpace augmented


,idx,label,sentence,score,lambda,sbh
0,63,anatomy,Cartilage protected the knee from constant fri...,0.006577,0.010118,0.000000
1,61,anatomy,Signals travelled along the neuron toward the ...,0.035538,0.000000,0.101537
2,56,emotions,Their success brought relief more than excitem...,0.038253,0.009802,0.091093
3,49,emotions,A quiet sense of dread settled over the empty ...,0.041864,0.020659,0.081245
4,62,anatomy,The surgeon examined the ventricle on the scan.,0.045307,0.015956,0.099814


## 10. Stability across random subcorpus samples

Notebook 02 suggested the semantic effect is not only visual. Here we add a simple robustness check.

We repeatedly subsample the sentence corpus, rebuild the PCA/KDE basin score and ArrowSpace map on each subcorpus, then compare minima overlap. The purpose is not to estimate a formal uncertainty interval, but to check whether augmentation is systematically more stable than the geometry-only baseline.


In [17]:
def rebuild_scores_on_subset(idxs):
    Xs = X[idxs]
    pca_s = PCA(n_components=2, random_state=42)
    X2s = pca_s.fit_transform(Xs)
    kde_s = gaussian_kde(X2s.T, bw_method=KDE_BANDWIDTH)
    sbh_s = -kde_s(X2s.T)
    sbh_s = (sbh_s - sbh_s.min()) / (sbh_s.max() - sbh_s.min() + 1e-12)
    lam_s, _, _ = build_lambda_map(Xs)
    return X2s, sbh_s, lam_s


def minima_from_scores(alpha, sbh, lam, q=BASIN_PERCENTILE):
    s = alpha * sbh + (1 - alpha) * lam
    th = np.percentile(s, q)
    return np.where(s <= th)[0]

stability_rows = []
seed_sets_bh = []
seed_sets_aug = []

for seed in range(N_STABILITY_SEEDS):
    rng = np.random.default_rng(100 + seed)
    subset = np.sort(rng.choice(np.arange(N_ITEMS), size=max(int(0.8 * N_ITEMS), 24), replace=False))
    _, sbh_s, lam_s = rebuild_scores_on_subset(subset)
    bh_s_local = minima_from_scores(1.0, sbh_s, lam_s)
    aug_s_local = minima_from_scores(ALPHA_HIGHLIGHT, sbh_s, lam_s)
    bh_global = subset[bh_s_local]
    aug_global = subset[aug_s_local]
    seed_sets_bh.append(set(map(int, bh_global)))
    seed_sets_aug.append(set(map(int, aug_global)))
    stability_rows.append({
        "seed": seed,
        "n_subset": len(subset),
        "bh_purity": supervised_purity(np.array(sorted(bh_global))),
        "aug_purity": supervised_purity(np.array(sorted(aug_global))),
        "bh_meanlam": float(LAM[list(bh_global)].mean()),
        "aug_meanlam": float(LAM[list(aug_global)].mean()),
    })


def mean_pairwise_jaccard(sets_):
    vals = []
    for i in range(len(sets_)):
        for j in range(i + 1, len(sets_)):
            a, b = sets_[i], sets_[j]
            vals.append(len(a & b) / (len(a | b) + 1e-12))
    return float(np.mean(vals)) if vals else np.nan

stability_df = pd.DataFrame(stability_rows)
stability_df.to_csv("output__03/stability_summary.csv", index=False)

mean_j_bh = mean_pairwise_jaccard(seed_sets_bh)
mean_j_aug = mean_pairwise_jaccard(seed_sets_aug)
print("Stability summary")
display(stability_df)
print(f"Mean pairwise Jaccard across subsamples — BH:  {mean_j_bh:.3f}")
print(f"Mean pairwise Jaccard across subsamples — AUG: {mean_j_aug:.3f}")


Stability summary


,seed,n_subset,bh_purity,aug_purity,bh_meanlam,aug_meanlam
0,0,76,0.3125,0.2500,0.107647,0.021377
1,1,76,0.3125,0.3125,0.125130,0.028170
2,2,76,0.3125,0.3750,0.084214,0.037209
3,3,76,0.2500,0.2500,0.119999,0.022514
4,4,76,0.2500,0.3125,0.098179,0.032580
5,5,76,0.3125,0.2500,0.070941,0.039314
6,6,76,0.3125,0.3125,0.126980,0.037579
7,7,76,0.2500,0.3125,0.156948,0.036433


Mean pairwise Jaccard across subsamples — BH:  0.146
Mean pairwise Jaccard across subsamples — AUG: 0.183


## 11. Consolidated metric summary

We compare three canonical conditions again:

- `alpha = 1.00` pure BasinHop geometry;
- `alpha = 0.35` canonical ArrowSpace augmentation;
- `alpha = 0.00` pure ArrowSpace typicality.


In [18]:
def format_row(label, alpha):
    idxs, _ = minima_set(alpha)
    r = eval_minima(idxs)
    jacc = len(np.intersect1d(BH_MINIMA, idxs)) / len(np.union1d(BH_MINIMA, idxs))
    return {
        "Condition": label,
        "alpha": alpha,
        "n": r["n"],
        "purity": round(r["purity"], 3),
        "coherence": round(r["coherence"], 4),
        "mean_lambda": round(r["meanlam"], 4),
        "dominant_field": r["dominant"],
        "Jaccard_vs_BH": round(jacc, 3),
    }

summary_df = pd.DataFrame([
    format_row("Pure BasinHop", 1.0),
    format_row(f"ArrowSpace-augmented ({ALPHA_HIGHLIGHT:.2f})", ALPHA_HIGHLIGHT),
    format_row("Pure ArrowSpace", 0.0),
])

display(summary_df)
summary_df.to_csv("output__03/metric_summary.csv", index=False)

best_purity_i = int(np.argmax(purities))
best_coh_i = int(np.argmax(coherences))
print(f"Best alpha by purity:    {alphas[best_purity_i]:.2f} (purity={purities[best_purity_i]:.3f})")
print(f"Best alpha by coherence: {alphas[best_coh_i]:.2f} (coherence={coherences[best_coh_i]:.4f})")


,Condition,alpha,n,purity,coherence,mean_lambda,dominant_field,Jaccard_vs_BH
0,Pure BasinHop,1.00,20,0.40,0.0964,0.0598,anatomy,1.000
1,ArrowSpace-augmented (0.35),0.35,20,0.35,0.0995,0.0298,anatomy,0.600
2,Pure ArrowSpace,0.00,20,0.25,0.1050,0.0081,anatomy,0.176


Best alpha by purity:    0.25 (purity=0.400)
Best alpha by coherence: 0.45 (coherence=0.1060)


## 12. Visualise basins, representatives, and the control surface

The figures mirror notebook 02:

1. sentence corpus in PCA-2D by field;
2. vanilla BasinHop minima overlay;
3. ArrowSpace-augmented minima overlay;
4. `alpha` sweep curves;
5. optional UMAP view;
6. stability comparison.


In [19]:
FIELD_TO_INT = {f: i for i, f in enumerate(FIELD_NAMES)}
FIELD_COLORS = {
    "astronomy": "#4C72B0",
    "programming": "#55A868",
    "cooking": "#DD8452",
    "finance": "#C44E52",
    "emotions": "#8172B3",
    "anatomy": "#937860",
    "music": "#DA8BC3",
    "geography": "#64B5CD",
}
colors = [FIELD_COLORS[l] for l in LABELS]


def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=170, bbox_inches="tight")
    plt.close()


# Figure 1 — PCA scatter by field
fig, ax = plt.subplots(figsize=(9, 7))
for field in FIELD_NAMES:
    m = LABELS == field
    ax.scatter(X2[m, 0], X2[m, 1], s=36, alpha=0.78, color=FIELD_COLORS[field], label=field)
if P_boundary is not None:
    ax.scatter(X2_boundary[:, 0], X2_boundary[:, 1], s=80, marker='x', color='black', linewidths=1.4, label='boundary')
ax.set_title("Sentence corpus in PCA-2D")
ax.set_xlabel("PCA-1")
ax.set_ylabel("PCA-2")
ax.legend(ncol=3, frameon=False, fontsize=9)
savefig("output__03/fig1_pca_sentence_fields.png")

# Figure 2 — vanilla minima overlay
fig, ax = plt.subplots(figsize=(9, 7))
mask = np.ones(N_ITEMS, dtype=bool)
mask[BH_MINIMA] = False
ax.scatter(X2[mask, 0], X2[mask, 1], s=26, alpha=0.22, color="#bdbdbd")
ax.scatter(X2[BH_MINIMA, 0], X2[BH_MINIMA, 1], s=70, alpha=0.92,
           c=[FIELD_COLORS[l] for l in LABELS[BH_MINIMA]], edgecolor='black', linewidth=0.35)
ax.set_title("Vanilla BasinHop minima in PCA-2D")
ax.set_xlabel("PCA-1")
ax.set_ylabel("PCA-2")
savefig("output__03/fig2_bh_minima_overlay.png")

# Figure 3 — augmented overlay
fig, ax = plt.subplots(figsize=(9, 7))
bg = ~(np.isin(np.arange(N_ITEMS), BH_MINIMA) | np.isin(np.arange(N_ITEMS), AUG_MINIMA))
vanonly = np.setdiff1d(BH_MINIMA, AUG_MINIMA)
augonly = np.setdiff1d(AUG_MINIMA, BH_MINIMA)
both = np.intersect1d(BH_MINIMA, AUG_MINIMA)
ax.scatter(X2[bg, 0], X2[bg, 1], s=24, alpha=0.20, color="#cccccc", label="background")
ax.scatter(X2[vanonly, 0], X2[vanonly, 1], s=62, alpha=0.90, color="#DD8452", label="vanilla only")
ax.scatter(X2[augonly, 0], X2[augonly, 1], s=62, alpha=0.90, color="#9467BD", label="augmented only")
ax.scatter(X2[both, 0], X2[both, 1], s=72, alpha=0.96, color="#2CA02C", label="both")
ax.set_title(f"BasinHop vs ArrowSpace-augmented minima (alpha={ALPHA_HIGHLIGHT:.2f})")
ax.set_xlabel("PCA-1")
ax.set_ylabel("PCA-2")
ax.legend(frameon=False)
savefig("output__03/fig3_bh_aug_overlay.png")

# Figure 4 — alpha sweep
fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))
axes[0].plot(alphas, purities, marker='o', color='steelblue')
axes[0].axvline(ALPHA_HIGHLIGHT, color='crimson', linestyle='--')
axes[0].set_title('Purity vs alpha')
axes[0].set_xlabel('alpha')
axes[0].set_ylim(0, 1.05)

axes[1].plot(alphas, coherences, marker='o', color='darkorange')
axes[1].axvline(ALPHA_HIGHLIGHT, color='crimson', linestyle='--')
axes[1].set_title('Coherence vs alpha')
axes[1].set_xlabel('alpha')

axes[2].plot(alphas, meanlams, marker='o', color='forestgreen')
axes[2].axvline(ALPHA_HIGHLIGHT, color='crimson', linestyle='--')
axes[2].set_title('Mean lambda vs alpha')
axes[2].set_xlabel('alpha')

axes[3].plot(alphas, nboundary, marker='o', color='slategray')
axes[3].axvline(ALPHA_HIGHLIGHT, color='crimson', linestyle='--')
axes[3].set_title('Boundary members vs alpha')
axes[3].set_xlabel('alpha')

savefig("output__03/fig4_alpha_sweep_curves.png")

# Figure 5 — optional UMAP
if X_umap is not None:
    fig, ax = plt.subplots(figsize=(9, 7))
    for field in FIELD_NAMES:
        m = LABELS == field
        ax.scatter(X_umap[m, 0], X_umap[m, 1], s=36, alpha=0.78, color=FIELD_COLORS[field], label=field)
    if X_umap_boundary is not None:
        ax.scatter(X_umap_boundary[:, 0], X_umap_boundary[:, 1], s=80, marker='x', color='black', linewidths=1.4, label='boundary')
    ax.set_title('Sentence corpus in UMAP-2D')
    ax.set_xlabel('UMAP-1')
    ax.set_ylabel('UMAP-2')
    ax.legend(ncol=3, frameon=False, fontsize=9)
    savefig("output__03/fig5_umap_sentence_fields.png")

# Figure 6 — stability bars
fig, ax = plt.subplots(figsize=(8, 4.8))
means = [stability_df['bh_purity'].mean(), stability_df['aug_purity'].mean(), mean_j_bh, mean_j_aug]
labels_bar = ['BH purity', 'AUG purity', 'BH stability', 'AUG stability']
colors_bar = ['#DD8452', '#9467BD', '#F2A65A', '#7A68A6']
ax.bar(labels_bar, means, color=colors_bar)
for i, v in enumerate(means):
    ax.text(i, v + 0.01, f"{v:.2f}", ha='center', va='bottom', fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_title('Purity and stability summary')
savefig("output__03/fig6_stability_summary.png")

print("Saved figures to output__03/")


Saved figures to output__03/


## 13. Save sentence-level inspection tables

These CSV artefacts are the useful downstream outputs of the notebook:

- metric summary;
- alpha sweep table;
- representative sentences;
- stability summary.


In [20]:
sweep_df = pd.DataFrame(sweep_results)
sweep_df.to_csv("output__03/alpha_sweep.csv", index=False)

sentence_table = pd.DataFrame({
    "idx": np.arange(N_ITEMS),
    "label": LABELS,
    "sentence": sentences,
    "pca1": X2[:, 0],
    "pca2": X2[:, 1],
    "sbh_norm": SBH_NORM,
    "lambda": LAM,
    f"score_aug_{ALPHA_HIGHLIGHT:.2f}": augmented_score(ALPHA_HIGHLIGHT),
    "is_bh_minima": np.isin(np.arange(N_ITEMS), BH_MINIMA),
    "is_aug_minima": np.isin(np.arange(N_ITEMS), AUG_MINIMA),
})
if X_umap is not None:
    sentence_table["umap1"] = X_umap[:, 0]
    sentence_table["umap2"] = X_umap[:, 1]

sentence_table.to_csv("output__03/sentence_basin_table.csv", index=False)
print("Saved CSV artefacts: alpha_sweep.csv, metric_summary.csv, representatives_*.csv, stability_summary.csv, sentence_basin_table.csv")


Saved CSV artefacts: alpha_sweep.csv, metric_summary.csv, representatives_*.csv, stability_summary.csv, sentence_basin_table.csv


## 14. Take-aways

If the notebook behaves like notebook 02, the pattern to watch is:

- vanilla minima remain geometrically plausible but semantically mixed;
- ArrowSpace lowers mean `lambda` inside the minima set and often raises purity;
- representative sentences become easier to interpret because they sit closer to semantic field cores;
- boundary sentences become a useful stress test for how much the basin criterion tolerates ambiguity.

That completes the sentence-corpus extension of the notebook series.

The next natural step after this notebook is either:

1. **cross-model basin agreement** — repeat the same corpus with multiple encoders and compare minima overlap; or
2. **cross-layer basin tracking** — derive embeddings from multiple model layers and follow stable basins across depth.
